딥러닝은 **종단간 머신러닝** 이라고도 한다. 데이터부터 출력까지 사람의 손을 거치지 않는 딥러닝이기 때문에.. 

이번 장에서는 경사하강법을 통해 손실함수가 줄어드는 방향으로 신경망 가중치를 조절해봅시다

In [4]:
import numpy as np

def sum_squares_error(y, t):
    return 0.5 * np.sum((y-t)**2)

크로스(교차) 엔트로피? -> 정답인 클래스의 출력만 집중해서 손실 함수 측정

정답이 아니면 얼마나 틀린지 관심 없다.

***ln***쓰는 이유는 출력이 1.0 이하 + 최종적으로 sum에 - 붙이는거 생각하고 그래프 예상해보면..

In [8]:
def CE(y, t):
    verysmall = 1e-7
    return -np.sum(t*np.log(y+verysmall)) #verysmall값을 통해 0이 들어가는 일은 없도록 한다..

In [9]:
t = [0,1,0,0,0] #5개 클래스 중 정답은 1인 원 핫 인코딩
y = [0.1, 0.1, 0.5, 0.1, 0.2]

print(sum_squares_error(np.array(y), np.array(t)))
print(CE(np.array(y), np.array(t)))
   

0.56
2.302584092994546


In [10]:
y = [0.1, 0.8, 0.1, 0.1, 0]

print(sum_squares_error(np.array(y), np.array(t)))
print(CE(np.array(y), np.array(t)))

0.03499999999999999
0.22314342631421757


In [11]:
y = [0.0009, 0.9991, 0,0,0]

print(sum_squares_error(np.array(y), np.array(t)))
print(CE(np.array(y), np.array(t)))

8.100000000000107e-07
0.0009003051530881438


***미니 배치***

전체 데이터셋 훈련 데이터를 다 쓰긴 좀 그러니까 데이터 일부를 추려 전체의 근사치로 활용하는 것

여러번의 미니 배치 학습에서 얻은 CE를 전부 더한 후 배치 수 만큼 전부 나누면 훈련 데이터 전반에 대한 CE 등장!

In [12]:
import sys, os
from mnist import load_mnist

sys.path.insert(0, os.getcwd())

(x_train, t_train), (x_test, t_test) = \
    load_mnist(flatten = True, normalize = False, one_hot_label=True)

/Users/jeonghowon/Desktop/JourneyToLLAMA/Step1_MitDeep/mnist.py:109: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dataset = pickle.load(f)


In [13]:
print(x_train.shape)
print(x_test.shape)

(60000, 784)
(10000, 784)


In [ ]:
train_size = x_train.shape[0]
batch_size = 10 #배치 한개는 
batch_mask = np.random.choice(train_size, batch_size) #train_size의 범위 내에서 batch_size만큼 뽑아서 array
print(type(batch_mask))

x_batch = x_train[batch_mask]
t_batch = t_train[batch_mask] 
#이 Fancy Indexing은 numpy일때만 가능합니다..

x_batch.shape

<class 'numpy.ndarray'>


(10, 784)

In [ ]:
def CEbatch(y, t):
    if y.dim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)

    batch_size = y.shape[0]
    return CE(y,t)/batch_size

학습 데이터 배치 사이즈 = 1 -> 데이터 하나가 입력 -> 출력도 하나 : 10차원짜리 벡터 하나 나옴 (10, )
-> y.shape[0]로 배치 사이즈 읽는데 ***벡터***의 shape는 10이다 -> 데이터는 하난데 모르고 10으로 나누게 됨

따라서 2차원 데이터에 한개짜리 벡터가 있는 것으로 수정하는 것

반면에 2개 이상이면 출력은 (2,10)부터 시작..

In [18]:
reshape_test = np.array([1,2,3,4,5,6,7,8,9])
print(reshape_test.reshape(1, reshape_test.size))
print(reshape_test.reshape(-1, 3))

[[1 2 3 4 5 6 7 8 9]]
[[1 2 3]
 [4 5 6]
 [7 8 9]]


Reshape? 메모리에 놓여진 데이터를 어떻게 바라볼 것인가

reshape(a, b) -> 데이터를 b개씩 끊어서 a줄로 읽어라

ps. 

reshape(a, -1) -> 알아서 데이터를 a줄로 묶어서 읽기

reshape(-1, b) -> 알아서 데이터를 b개씩 끊어 읽기

In [17]:
print(reshape_test)

[1 2 3 4 5 6 7 8 9]


++언제나 메모리 원래 주소 보므로 reshape한 상태에서 수정해도 원본 바뀐다는 것 주의. 데이터는 언제나 메모리상에 일직선으로 건재하기에 reshape 호출 자체로 데이터가 변하거나 할 수는 없지만 reshape 된 상태의 인덱싱을 적용해도 대응하는 데이터가 바뀌게 된다는 것.

In [27]:
#print(reshape_test.size)
print(reshape_test.ndim)

1


dim과 size의 차이를 기억합시다!